**EJEMPLO SECCIÓN 2.1.4. (FNN 784 -> 256 -> 10)**


In [ ]:
!pip install pytorch_lightning --quiet
!pip install ISLP --quiet
!pip install torchinfo --quiet

import numpy as np, pandas as pd
from matplotlib.pyplot import subplots
from sklearn.linear_model import \
(LinearRegression ,
LogisticRegression ,
Lasso)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from ISLP import load_data
from ISLP.models import ModelSpec as MS
from sklearn.model_selection import \
(train_test_split ,
GridSearchCV)

import torch
from torch import nn
from torch.optim import RMSprop
from torch.utils.data import TensorDataset

from torchmetrics import (MeanAbsoluteError ,
R2Score)
from torchinfo import summary
from torchvision.io import read_image

from pytorch_lightning import Trainer
from pytorch_lightning.loggers import CSVLogger

#SEMILLA PARA QUE SIEMPRE DE EL MISMO RESULTADO (SEMILLA = 0 EN ESTE CASO)
from pytorch_lightning import seed_everything
seed_everything(0, workers=True)
torch.use_deterministic_algorithms(True, warn_only=True)

from torchvision.datasets import MNIST
from torchvision.models import (resnet50 ,
ResNet50_Weights)
from torchvision.transforms import (Resize ,
Normalize ,
CenterCrop ,
ToTensor)

from ISLP.torch import rec_num_workers
from ISLP.torch import SimpleDataModule
from ISLP.torch import SimpleModule
from ISLP.torch import ErrorTracker

In [ ]:
from ISLP.torch import rec_num_workers

max_num_workers = rec_num_workers()


(mnist_train ,
mnist_test) = [MNIST(root='data',
train=train ,
download=True ,
transform=ToTensor())
for train in [True , False]]
mnist_train

In [ ]:
from ISLP.torch import SimpleDataModule

mnist_dm = SimpleDataModule(mnist_train ,
mnist_test ,
validation=0.2,
num_workers=max_num_workers ,
batch_size =256)

for idx , (X_ ,Y_) in enumerate(mnist_dm.train_dataloader()):
    print('X: ', X_.shape)
    print('Y: ', Y_.shape)
    if idx >= 1:
        break

class MNISTModel(nn.Module):
  def __init__(self):
    super(MNISTModel , self).__init__()
    self.layer1 = nn.Sequential(
      nn.Flatten(),
      nn.Linear(28*28, 256),
      nn.ReLU(),
      nn.Dropout (0.4))
    self._forward = nn.Sequential(
      self.layer1 ,
      nn.Linear(256, 10))

  def forward(self , x):
    return self._forward(x)

In [ ]:
mnist_model = MNISTModel()

summary(mnist_model ,
        input_data=X_,
        col_names=['input_size',
                    'output_size',
                    'num_params'])

In [ ]:
mnist_module = SimpleModule.classification(mnist_model, num_classes=10)
mnist_logger = CSVLogger('logs', name='MNIST')

mnist_trainer = Trainer(deterministic=True ,
                        max_epochs=30,
                        logger=mnist_logger ,
                        callbacks=[ErrorTracker()])
mnist_trainer.fit(mnist_module ,
                  datamodule=mnist_dm)

In [ ]:
def summary_plot(results ,
                  ax,
                  col='loss',
                  valid_legend='Tasa de acierto en validación',
                  training_legend='Tasa de acierto en entrenamiento',
                  ylabel='Loss',
                  fontsize=20):
    for (column ,
          color ,
          label) in zip([f'train_{col}_epoch',
                        f'valid_{col}'],
                        ['blue',
                        'orange'],
                        [training_legend ,
                        valid_legend]):
        results.dropna(subset=[column]).plot(x='epoch',
                                              y=column ,
                                              label=label ,
                                              marker="none",
                                              color=color ,
                                              ax=ax)
    ax.set_xlabel('Épocas')
    ax.set_ylabel(ylabel)
    return ax

In [ ]:
mnist_results = pd.read_csv(mnist_logger.experiment.
  metrics_file_path)
fig , ax = subplots(1, 1, figsize=(6, 6))
summary_plot(mnist_results ,
              ax,
              col='accuracy',
              ylabel='AccTaauracy')
ax.set_ylim([0.5, 1])
ax.set_ylabel('Tasa de acierto')
ax.set_title("MNIST FNN con una única capa oculta - Curvas de aprendizaje")
ax.set_xticks(np.linspace(0, 30, 7).astype(int));

mnist_trainer.test(mnist_module ,
datamodule=mnist_dm)



*   Con con seed = 0 da una tasa de precisión de 97,31% (Con dropout)





**EJEMPLO SECCIÓN 2.2.3. (FNN 784 -> 256 -> 128 -> 10)**


In [ ]:
!pip install pytorch_lightning --quiet
!pip install ISLP --quiet
!pip install torchinfo --quiet

import numpy as np, pandas as pd
from matplotlib.pyplot import subplots
from sklearn.linear_model import \
(LinearRegression ,
LogisticRegression ,
Lasso)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from ISLP import load_data
from ISLP.models import ModelSpec as MS
from sklearn.model_selection import \
(train_test_split ,
GridSearchCV)

import torch
from torch import nn
from torch.optim import RMSprop
from torch.utils.data import TensorDataset

from torchmetrics import (MeanAbsoluteError ,
R2Score)
from torchinfo import summary
from torchvision.io import read_image

from pytorch_lightning import Trainer
from pytorch_lightning.loggers import CSVLogger

#SEMILLA PARA QUE SIEMPRE DE EL MISMO RESULTADO (SEMILLA = 0 EN ESTE CASO)
from pytorch_lightning import seed_everything
seed_everything(0, workers=True)
torch.use_deterministic_algorithms(True, warn_only=True)

from torchvision.datasets import MNIST
from torchvision.models import (resnet50 ,
ResNet50_Weights)
from torchvision.transforms import (Resize ,
Normalize ,
CenterCrop ,
ToTensor)

from ISLP.torch import rec_num_workers
from ISLP.torch import SimpleDataModule
from ISLP.torch import SimpleModule
from ISLP.torch import ErrorTracker

In [ ]:
from ISLP.torch import rec_num_workers

max_num_workers = rec_num_workers()


(mnist_train ,
mnist_test) = [MNIST(root='data',
train=train ,
download=True ,
transform=ToTensor())
for train in [True , False]]
mnist_train

In [ ]:
from ISLP.torch import SimpleDataModule

mnist_dm = SimpleDataModule(mnist_train ,
mnist_test ,
validation=0.2,
num_workers=max_num_workers ,
batch_size =256)

In [ ]:
for idx , (X_ ,Y_) in enumerate(mnist_dm.train_dataloader()):
    print('X: ', X_.shape)
    print('Y: ', Y_.shape)
    if idx >= 1:
        break

In [ ]:
class MNISTModel(nn.Module):
  def __init__(self):
    super(MNISTModel , self).__init__()
    self.layer1 = nn.Sequential(
      nn.Flatten(),
      nn.Linear(28*28, 256),
      nn.ReLU(),
      nn.Dropout (0.4))
    self.layer2 = nn.Sequential(
      nn.Linear(256, 128),
      nn.ReLU(),
      nn.Dropout (0.3))
    self._forward = nn.Sequential(
      self.layer1 ,
      self.layer2 ,
      nn.Linear(128, 10))

  def forward(self , x):
    return self._forward(x)

In [ ]:
mnist_model = MNISTModel()

In [ ]:
mnist_model(X_).size()

In [ ]:
summary(mnist_model ,
        input_data=X_,
        col_names=['input_size',
                    'output_size',
                    'num_params'])

In [ ]:
mnist_module = SimpleModule.classification(mnist_model, num_classes=10)
mnist_logger = CSVLogger('logs', name='MNIST')

In [ ]:
mnist_trainer = Trainer(deterministic=True ,
                        max_epochs=30,
                        logger=mnist_logger ,
                        callbacks=[ErrorTracker()])
mnist_trainer.fit(mnist_module ,
                  datamodule=mnist_dm)

In [ ]:
def summary_plot(results ,
                  ax,
                  col='loss',
                  valid_legend='Tasa de acierto en validación',
                  training_legend='Tasa de acierto en entrenamiento',
                  ylabel='Loss',
                  fontsize=20):
    for (column ,
          color ,
          label) in zip([f'train_{col}_epoch',
                        f'valid_{col}'],
                        ['blue',
                        'orange'],
                        [training_legend ,
                        valid_legend]):
        results.dropna(subset=[column]).plot(x='epoch',
                                              y=column ,
                                              label=label ,
                                              marker="none",
                                              color=color ,
                                              ax=ax)
    ax.set_xlabel('Épocas')
    ax.set_ylabel(ylabel)
    return ax

In [ ]:
mnist_results = pd.read_csv(mnist_logger.experiment.
  metrics_file_path)
fig , ax = subplots(1, 1, figsize=(6, 6))
summary_plot(mnist_results ,
              ax,
              col='accuracy',
              ylabel='AccTaauracy')
ax.set_ylim([0.5, 1])
ax.set_ylabel('Tasa de acierto')
ax.set_title("MNIST FNN multicapa - Curvas de aprendizaje")
ax.set_xticks(np.linspace(0, 30, 7).astype(int));

mnist_trainer.test(mnist_module ,
datamodule=mnist_dm)




*   Con seed = 0 da una tasa de acierto del 96,65% (Con DROPOUT)








In [ ]:
print(mnist_results.columns.tolist())
print(mnist_results.head(20))

**RED NEURONAL CONVOLUCIONAL MNIST**

In [ ]:
!pip install pytorch_lightning --quiet
!pip install ISLP --quiet
!pip install torchinfo --quiet

import numpy as np, pandas as pd
from matplotlib.pyplot import subplots
from sklearn.linear_model import \
(LinearRegression ,
LogisticRegression ,
Lasso)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from ISLP import load_data
from ISLP.models import ModelSpec as MS
from sklearn.model_selection import \
(train_test_split ,
GridSearchCV)

import torch
from torch import nn
from torch.optim import RMSprop
from torch.utils.data import TensorDataset

from torchmetrics import (MeanAbsoluteError ,
R2Score)
from torchinfo import summary
from torchvision.io import read_image

from pytorch_lightning import Trainer
from pytorch_lightning.loggers import CSVLogger

#SEMILLA PARA QUE SIEMPRE DE EL MISMO RESULTADO (SEMILLA = 0 EN ESTE CASO)
from pytorch_lightning import seed_everything
seed_everything(0, workers=True)
torch.use_deterministic_algorithms(True, warn_only=True)

from torchvision.datasets import MNIST
from torchvision.models import (resnet50 ,
ResNet50_Weights)
from torchvision.transforms import (Resize ,
Normalize ,
CenterCrop ,
ToTensor)

from ISLP.torch import rec_num_workers
from ISLP.torch import SimpleDataModule
from ISLP.torch import SimpleModule
from ISLP.torch import ErrorTracker

In [ ]:
from ISLP.torch import rec_num_workers

max_num_workers = rec_num_workers()


(mnist_train ,
mnist_test) = [MNIST(root='data',
                    train=train ,
                    download=True ,
                    transform=ToTensor())
              for train in [True , False]]
mnist_train

In [ ]:
transform = ToTensor()

#Utilizo esto para convertir los datos de MNIST a tensores
mnist_train_X = torch.stack([transform(x) for x in mnist_train.data.numpy()])
mnist_test_X  = torch.stack([transform(x) for x in mnist_test.data.numpy()])

mnist_train = TensorDataset(mnist_train_X,
                            torch.tensor(mnist_train.targets))
mnist_test  = TensorDataset(mnist_test_X,
                            torch.tensor(mnist_test.targets))

In [ ]:
#Sirve para visualizar algunas imágenes del MNIST
fig, axes = subplots(5, 5, figsize=(6, 6))
rng = np.random.default_rng(4)
indices = rng.choice(np.arange(len(mnist_train)), 25,
                     replace=False).reshape((5, 5))
for i in range(5):
    for j in range(5):
        idx = indices[i, j]
        # mnist_train_X tiene forma (N, 1, 28, 28); utilizo squeeze para eliminar el canal ya que son imágenes en escala de grises
        axes[i, j].imshow(mnist_train_X[idx].squeeze(), cmap='gray',
                          interpolation=None)
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

In [ ]:
from ISLP.torch import rec_num_workers

max_num_workers = rec_num_workers()

mnist_dm = SimpleDataModule(mnist_train ,
                            mnist_test ,
                            validation=0.2,
                            num_workers=max_num_workers ,
                            batch_size =128)

In [ ]:
for idx , (X_ ,Y_) in enumerate(mnist_dm.train_dataloader()):
    print('X: ', X_.shape)
    print('Y: ', Y_.shape)
    if idx >= 1:
        break

In [ ]:
class BuildingBlock(nn.Module):
    def __init__(self,
                 in_channels,
                 out_channels):
        super(BuildingBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels=in_channels,
                              out_channels=out_channels,
                              kernel_size=(3, 3),
                              padding='same')
        self.activation = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=(2, 2))

    def forward(self, x):
        return self.pool(self.activation(self.conv(x)))

In [ ]:
class MNISTModel(nn.Module):
    def __init__(self):
        super(MNISTModel, self).__init__()
        sizes = [(1, 32),
                 (32, 64)]
        self.conv = nn.Sequential(*[BuildingBlock(in_, out_)
                                    for in_, out_ in sizes])
        self.output = nn.Sequential(nn.Dropout(0.5),
                                    nn.Linear(64 * 7 * 7, 256),
                                    nn.ReLU(),
                                    nn.Linear(256, 10))

    def forward(self, x):
        val = self.conv(x)
        val = torch.flatten(val, start_dim=1)
        return self.output(val)

In [ ]:
mnist_model = MNISTModel()
summary(mnist_model,
        input_data=X_,
        col_names=['input_size',
                   'output_size',
                   'num_params'])

In [ ]:
mnist_optimizer = RMSprop(mnist_model.parameters(), lr=0.001)
mnist_module    = SimpleModule.classification(mnist_model,
                                              optimizer=mnist_optimizer,
                                              num_classes=10)
mnist_logger    = CSVLogger('logs', name='MNIST')

mnist_trainer = Trainer(deterministic=True,
                        max_epochs=30,
                        logger=mnist_logger,
                        callbacks=[ErrorTracker()])
mnist_trainer.fit(mnist_module,
                  datamodule=mnist_dm)

In [ ]:
def summary_plot(results ,
                  ax,
                  col='loss',
                  valid_legend='Tasa de acierto en validación',
                  training_legend='Tasa de acierto en entrenamiento',
                  ylabel='Loss',
                  fontsize=20):
    for (column ,
          color ,
          label) in zip([f'train_{col}_epoch',
                        f'valid_{col}'],
                        ['blue',
                        'orange'],
                        [training_legend ,
                        valid_legend]):
        results.dropna(subset=[column]).plot(x='epoch',
                                              y=column ,
                                              label=label ,
                                              marker="none",
                                              color=color ,
                                              ax=ax)
    ax.set_xlabel('Épocas')
    ax.set_ylabel(ylabel)
    return ax

In [ ]:
mnist_trainer.test(mnist_module, datamodule=mnist_dm)

log_path = mnist_logger.experiment.metrics_file_path
mnist_results = pd.read_csv(log_path)

fig, ax = subplots(1, 1, figsize=(6, 6))
summary_plot(mnist_results,
             ax,
             col='accuracy',
             ylabel='Accuracy')
ax.set_xticks(np.linspace(0, 30, 7).astype(int))
ax.set_ylabel('Tasa de Acierto')
ax.set_ylim([0, 1])
ax.set_title('CNN MNIST – Curvas de aprendizaje')
ax.legend(loc='lower right')

Con seed=0 da una tasa de acierto del 99,34% con RSProp

**GRÁFICAS UTILIZADAS EN EL TFG**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#ReLU
x = np.linspace(-3, 3, 500)
y = np.maximum(0, x)

fig, ax = plt.subplots()
ax.plot(x, y, 'r', linewidth=2)

estilo_ejes(ax)
ax.set_xlim(-3, 3)
ax.set_ylim(0, 3)
plt.show()

In [ ]:
#Leaky ReLU

# Función Leaky ReLU
def f(x):
    return np.where(x <= 0, 0.1*x, x)

x = np.linspace(-3, 3, 500)
y = f(x)

# Gráfica
fig, ax = plt.subplots()
ax.plot(x, y, 'r', linewidth=2)


fig, ax = plt.subplots()
ax.plot(x, y, 'r', linewidth=2)

estilo_ejes(ax)
ax.set_xlim(-3, 3)
ax.set_ylim(-1, 3)
plt.show()

In [ ]:
#FUNCIÓN RADIAL DE GAUSS
x = np.linspace(-3, 3, 500)
y = np.exp(-x**2)

fig, ax = plt.subplots()
ax.plot(x, y, 'r', linewidth=2)

estilo_ejes(ax)
ax.set_xlim(-3, 3)
ax.set_ylim(0, 1.2)
plt.show()

In [ ]:
#SIGMOIDE
x = np.linspace(-10, 10, 500)
y = 1 / (1 + np.exp(-x))

fig, ax = plt.subplots()
ax.plot(x, y, 'r', linewidth=2)

estilo_ejes(ax)
ax.set_xlim(-10, 10)
ax.set_ylim(0, 1.1)

ax.axhline(1, linestyle='--', linewidth=1)
plt.show()

In [ ]:
#TANH
x = np.linspace(-3, 3, 500)
y = np.tanh(x)

fig, ax = plt.subplots()
ax.plot(x, y, 'r', linewidth=2)

estilo_ejes(ax)
ax.set_xlim(-3, 3)
ax.set_ylim(-1.2, 1.2)

# Líneas discontinuas
ax.axhline(1, linestyle='--', linewidth=1)
ax.axhline(-1, linestyle='--', linewidth=1)
plt.show()

In [ ]:
#BINARIA

fig, ax = plt.subplots(figsize=(6, 4))

x_neg = np.linspace(-2.2, 0, 100)
y_neg = np.zeros_like(x_neg)

x_pos = np.linspace(0.001, 2.2, 100)
y_pos = np.ones_like(x_pos)

ax.plot(x_neg, y_neg, color='red', linewidth=3.5)
ax.plot(x_pos, y_pos, color='red', linewidth=3.5)

ax.spines['left'].set_position('zero')
ax.spines['bottom'].set_position('zero')

ax.spines['right'].set_color('none')
ax.spines['top'].set_color('none')

ax.set_xlim(-2.4, 2.4)
ax.set_ylim(-0.2, 1.2)
plt.show()

In [ ]:
#GRÁFICA SIGMOIDE Y ReLU
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-5, 65, 500)

fig, ax = plt.subplots()

# ReLU
y_relu = np.maximum(0, x)/5
ax.plot(x, y_relu, 'r', linewidth=2, label='ReLU')

# Sigmoide
y_sigmoid = 1 / (1 + np.exp(-x))
ax.plot(x, y_sigmoid, 'b', linewidth=2, label='Sigmoide')
ax.axhline(1, linestyle='--', linewidth=1, color='b', alpha=0.4)

ax.set_xlim(-5, 5)
ax.set_ylim(0, 1)
ax.legend()
ax.set_xlabel("x")
ax.set_ylabel("g(x)")
plt.show()

**RED NEURONAL CONVOLUCIONAL CFAR100 (EJEMPLO LIBRO ISLP PARA IMITARLO EN EL MNIST)**

In [ ]:
!pip install pytorch_lightning --quiet
!pip install ISLP --quiet
!pip install torchinfo --quiet

import numpy as np, pandas as pd
from matplotlib.pyplot import subplots
from sklearn.linear_model import \
(LinearRegression ,
LogisticRegression ,
Lasso)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from ISLP import load_data
from ISLP.models import ModelSpec as MS
from sklearn.model_selection import \
(train_test_split ,
GridSearchCV)

import torch
from torch import nn
from torch.optim import RMSprop
from torch.utils.data import TensorDataset

from torchmetrics import (MeanAbsoluteError ,
R2Score)
from torchinfo import summary
from torchvision.io import read_image

from pytorch_lightning import Trainer
from pytorch_lightning.loggers import CSVLogger

#SEMILLA PARA QUE SIEMPRE DE EL MISMO RESULTADO (SEMILLA = 0 EN ESTE CASO)
from pytorch_lightning import seed_everything
seed_everything(0, workers=True)
torch.use_deterministic_algorithms(True, warn_only=True)

from torchvision.datasets import MNIST, CIFAR100
from torchvision.models import (resnet50 ,
ResNet50_Weights)
from torchvision.transforms import (Resize ,
Normalize ,
CenterCrop ,
ToTensor)

from ISLP.torch import rec_num_workers
from ISLP.torch import SimpleDataModule
from ISLP.torch import SimpleModule
from ISLP.torch import ErrorTracker

In [ ]:
(cifar_train ,
cifar_test) = [CIFAR100(root="data",
                        train=train ,
                        download=True)
            for train in [True , False]]

In [ ]:
transform = ToTensor()
cifar_train_X = torch.stack([transform(x) for x in
cifar_train.data])
cifar_test_X = torch.stack([transform(x) for x in
cifar_test.data])
cifar_train = TensorDataset(cifar_train_X ,
torch.tensor(cifar_train.targets))
cifar_test = TensorDataset(cifar_test_X ,
torch.tensor(cifar_test.targets))

In [ ]:
from ISLP.torch import rec_num_workers

max_num_workers = rec_num_workers()

cifar_dm = SimpleDataModule(cifar_train ,
                            cifar_test ,
                            validation=0.2,
                            num_workers=max_num_workers ,
                            batch_size =128)

In [ ]:
for idx , (X_ ,Y_) in enumerate(cifar_dm.train_dataloader()):
    print('X: ', X_.shape)
    print('Y: ', Y_.shape)
    if idx >= 1:
      break

In [ ]:
fig , axes = subplots(5, 5, figsize=(10,10))
rng = np.random.default_rng(4)
indices = rng.choice(np.arange(len(cifar_train)), 25,
                    replace=False).reshape((5,5))
for i in range(5):
    for j in range(5):
        idx = indices[i,j]
        axes[i,j].imshow(np.transpose(cifar_train[idx][0],
                          [1,2,0]),
                          interpolation=None)
        axes[i,j].set_xticks ([])
        axes[i,j].set_yticks ([])

In [ ]:
class BuildingBlock(nn.Module):
    def __init__(self ,
                  in_channels ,
                  out_channels):
        super(BuildingBlock , self).__init__()
        self.conv = nn.Conv2d(in_channels=in_channels ,
                              out_channels=out_channels ,
                              kernel_size=(3,3),
                              padding='same')
        self.activation = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=(2,2))

    def forward(self , x):
        return self.pool(self.activation(self.conv(x)))

In [ ]:
class CIFARModel(nn.Module):
    def __init__(self):
        super(CIFARModel , self).__init__()
        sizes = [(3,32),
                (32,64),
                (64,128),
                (128,256)]
        self.conv = nn.Sequential(*[BuildingBlock(in_ , out_)
                                  for in_ , out_ in sizes])
        self.output = nn.Sequential(nn.Dropout (0.5),
                                    nn.Linear(2*2*256, 512),
                                    nn.ReLU(),
                                    nn.Linear(512, 100))
    def forward(self , x):
        val = self.conv(x)
        val = torch.flatten(val , start_dim=1)
        return self.output(val)

In [ ]:
cifar_model = CIFARModel()
summary(cifar_model ,
        input_data=X_,
        col_names=['input_size',
                  'output_size',
                  'num_params'])

In [ ]:
cifar_optimizer = RMSprop(cifar_model.parameters(), lr=0.001)
cifar_module = SimpleModule.classification(cifar_model ,optimizer=cifar_optimizer, num_classes=100)
cifar_logger = CSVLogger('logs', name='CIFAR100')

In [ ]:
cifar_trainer = Trainer(deterministic=True ,
                        max_epochs=30,
                        logger=cifar_logger ,
                        callbacks=[ErrorTracker()])
cifar_trainer.fit(cifar_module ,
                  datamodule=cifar_dm)

In [ ]:
def summary_plot(results ,
                  ax,
                  col='loss',
                  valid_legend='Tasa de acierto en validación',
                  training_legend='Tasa de acierto en entrenamiento',
                  ylabel='Loss',
                  fontsize=20):
    for (column ,
          color ,
          label) in zip([f'train_{col}_epoch',
                        f'valid_{col}'],
                        ['blue',
                        'orange'],
                        [training_legend ,
                        valid_legend]):
        results.dropna(subset=[column]).plot(x='epoch',
                                              y=column ,
                                              label=label ,
                                              marker="none",
                                              color=color ,
                                              ax=ax)
    ax.set_xlabel('Épocas')
    ax.set_ylabel(ylabel)
    return ax

In [ ]:
log_path = cifar_logger.experiment.metrics_file_path
cifar_results = pd.read_csv(log_path)
fig , ax = subplots(1, 1, figsize=(6, 6))
summary_plot(cifar_results ,
              ax,
              col='accuracy',
              ylabel='Accuracy')
ax.set_xticks(np.linspace(0, 10, 6).astype(int))
ax.set_ylabel('Accuracy')
ax.set_ylim([0, 1]);

In [ ]:
cifar_trainer.test(cifar_module ,datamodule=cifar_dm)